# PDEs and Boundary Conditions

**Time: ~40 minutes**

We step up from ODEs to PDEs. We'll solve Burgers' equation from scratch:

$$u_t + u \cdot u_x = \nu \cdot u_{xx}$$

This introduces: spatial derivatives, boundary conditions, 2D collocation, and the challenge of nonlinear PDEs.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
print("Ready.")

## Burgers' Equation

**Domain:** `x in [-1, 1]`, `t in [0, 1]`

**Initial condition:** `u(x, 0) = -sin(pi * x)`

**Boundary conditions:** `u(-1, t) = u(1, t) = 0`

**Viscosity:** `nu = 0.01/pi` (low viscosity → sharp gradients form)

This is a canonical PDE benchmark for PINNs. The solution develops a steep gradient (near-shock) as the nonlinear advection term steepens the wave faster than viscosity can smooth it.

In [ ]:
nu = 0.01 / np.pi

# Show the initial condition
x_plot = np.linspace(-1, 1, 200)
plt.figure(figsize=(8, 4))
plt.plot(x_plot, -np.sin(np.pi * x_plot), 'b-', linewidth=2)
plt.xlabel('x'); plt.ylabel('u')
plt.title('Initial condition: u(x, 0) = -sin(pi*x)')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 1: The Network

Now we take **two inputs** `(x, t)` and output `u(x, t)`.

In [ ]:
class BurgersPINN(nn.Module):
    def __init__(self, hidden=64, layers=4):
        super().__init__()
        modules = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            modules += [nn.Linear(hidden, hidden), nn.Tanh()]
        modules.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*modules)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

model = BurgersPINN()
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## Step 2: Collocation Points

For a PDE in 2D, we need points in the interior, on the initial condition, and on the boundaries.

In [ ]:
N_physics = 5000   # interior collocation points
N_ic = 200         # initial condition points
N_bc = 200         # boundary condition points (each boundary)

# Interior: random (x, t) in [-1,1] x [0,1]
x_phys = (2 * torch.rand(N_physics, 1) - 1).requires_grad_(True)  # [-1, 1]
t_phys = torch.rand(N_physics, 1).requires_grad_(True)             # [0, 1]

# IC: t = 0, x in [-1, 1]
x_ic = (2 * torch.rand(N_ic, 1) - 1)
t_ic = torch.zeros(N_ic, 1)
u_ic = -torch.sin(np.pi * x_ic)  # u(x, 0) = -sin(pi*x)

# BC: x = -1 and x = 1, t in [0, 1]
t_bc = torch.rand(N_bc, 1)
x_bc_left = -torch.ones(N_bc, 1)   # x = -1
x_bc_right = torch.ones(N_bc, 1)   # x = +1

# Visualize the sampling
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_phys.detach(), t_phys.detach(), s=1, alpha=0.3, label=f'Interior ({N_physics})')
ax.scatter(x_ic.detach(), t_ic.detach(), s=10, c='red', label=f'IC ({N_ic})')
ax.scatter(x_bc_left.detach(), t_bc.detach(), s=10, c='green', label=f'BC left ({N_bc})')
ax.scatter(x_bc_right.detach(), t_bc.detach(), s=10, c='orange', label=f'BC right ({N_bc})')
ax.set_xlabel('x'); ax.set_ylabel('t')
ax.set_title('Collocation Points')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 3: Loss Functions

Three loss terms:
- **Physics**: `u_t + u * u_x - nu * u_xx = 0`
- **IC**: `u(x, 0) = -sin(pi*x)`
- **BC**: `u(-1, t) = u(1, t) = 0`

In [ ]:
def physics_loss(model):
    u = model(x_phys, t_phys)
    
    # First derivatives
    u_x = torch.autograd.grad(u, x_phys, torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t_phys, torch.ones_like(u), create_graph=True)[0]
    
    # Second derivative
    u_xx = torch.autograd.grad(u_x, x_phys, torch.ones_like(u_x), create_graph=True)[0]
    
    # Burgers residual: u_t + u * u_x - nu * u_xx = 0
    residual = u_t + u * u_x - nu * u_xx
    return torch.mean(residual**2)

def ic_loss(model):
    u_pred = model(x_ic, t_ic)
    return torch.mean((u_pred - u_ic)**2)

def bc_loss(model):
    u_left = model(x_bc_left, t_bc)
    u_right = model(x_bc_right, t_bc)
    return torch.mean(u_left**2) + torch.mean(u_right**2)

print("Loss functions defined.")

## Step 4: Training

We weight the IC more heavily to ensure it's satisfied early.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 10000

w_physics, w_ic, w_bc = 1.0, 20.0, 10.0

history = {"physics": [], "ic": [], "bc": [], "total": []}

for epoch in range(n_epochs):
    optimizer.zero_grad()
    
    lp = physics_loss(model)
    li = ic_loss(model)
    lb = bc_loss(model)
    total = w_physics * lp + w_ic * li + w_bc * lb
    
    total.backward()
    optimizer.step()
    
    history["physics"].append(lp.item())
    history["ic"].append(li.item())
    history["bc"].append(lb.item())
    history["total"].append(total.item())
    
    if epoch % 2000 == 0:
        print(f"Epoch {epoch:5d} | phys={lp.item():.3e} | ic={li.item():.3e} | bc={lb.item():.3e}")

print(f"\nFinal total loss: {history['total'][-1]:.4e}")

In [ ]:
# Loss history
fig, ax = plt.subplots(figsize=(8, 5))
for key in ["total", "physics", "ic", "bc"]:
    ax.semilogy(history[key], label=key, alpha=0.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Loss History")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 5: Visualize the Solution

In [ ]:
# Evaluate on a grid
nx, nt = 200, 100
x_grid = np.linspace(-1, 1, nx)
t_grid = np.linspace(0, 1, nt)
X, T = np.meshgrid(x_grid, t_grid)

x_flat = torch.tensor(X.flatten(), dtype=torch.float32).unsqueeze(1)
t_flat = torch.tensor(T.flatten(), dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    u_pred = model(x_flat, t_flat).numpy().reshape(nt, nx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot
c = axes[0].contourf(X, T, u_pred, levels=50, cmap='RdBu_r')
plt.colorbar(c, ax=axes[0])
axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
axes[0].set_title('PINN Solution u(x, t)')

# Time slices
for t_val in [0.0, 0.25, 0.5, 0.75, 1.0]:
    idx = int(t_val * (nt - 1))
    axes[1].plot(x_grid, u_pred[idx, :], label=f't = {t_val}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('u')
axes[1].set_title('Solution at Different Times')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("Watch the initial sine wave steepen into a near-shock at t ≈ 0.5-1.0.")
print("This is the nonlinear advection term in action.")

## What Made This Harder Than the ODE?

| Challenge | How we handled it |
|-----------|-------------------|
| **2 independent variables** (x, t) | Separate tensors, each with `requires_grad=True` |
| **Partial derivatives** (u_x, u_t, u_xx) | `autograd.grad` with the right input tensor |
| **Boundary conditions** | Separate loss terms for x = -1 and x = 1 |
| **Nonlinearity** (u * u_x) | No special treatment — autograd handles it |
| **Sharp gradients** | More collocation points, more epochs |
| **Loss balancing** | IC weighted 20x to ensure it's satisfied early |

The fundamental pattern is identical to the ODE case: network → autograd → residual → minimize.

## Exercises

1. **Higher viscosity**: Try `nu = 0.1/pi`. The solution should be smoother. Is it easier to learn?
2. **More collocation points**: Try 20,000 interior points. Better or just slower?
3. **Hard boundary conditions**: Instead of a BC loss, modify the network output:
   `u = (1-x²) * network(x,t) + g(x)` where `g(x) = -sin(pi*x) * (1-t)`. This satisfies BCs by construction.

## What's Next

**Notebook 06** covers the training tricks that matter in practice: loss weighting strategies, dealing with spectral bias, learning rate schedules, and the Ansatz trick.